## **Example usage of EnergyPod Calculator**

This notebook is a step by step guide through the different components of the EnergyPod Calculator. It shows you how to use the different components of the API as a whole, in the way that is also explained in [`docs/architecture/api_architecture.md`](../docs/architecture/api_architecture.md). The modules can also be used as stand alone units.

#### **Import necessary packages and modules**

In [2]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.application.baseload_profile import get_baseload_profile_from_business_category
from src.application.grid_tariffs import calculate_grid_tariff_large, calculate_grid_tariff_small
from src.application.payback_time import calculate_payback_time
from src.application.update_current_connection import UpdateCurrentConnection
from src.application.with_energy_pod import EnergyPodPower
from src.application.without_energy_pod import calculate_results_without_energypod
from src.application.yearly_savings import calculate_yearly_savings
from src.config import battery_price_per_kwh, cp_price_per_kw, first_day_of_year
from src.domain.energy_dto import EnergyDTOLargeConsumer, EnergyDTOSmallConsumer
from src.domain.grid_tariff import GridTariffDTOLargeConsumer, GridTariffDTOSmallConsumer
from src.domain.vehicle_info import VehicleInfoDTO, VehicleTypeInfoDTO

ModuleNotFoundError: No module named 'src'

#### **Example user input for vehicle information**

For each vehicle type (vans, boxtrucks and semitrailertrucks), define the number of vehicles (`nr_of_vehicles`), the power usage in kWh per km (`power_usage`) and the total number of km driven per year per vehicle (`annual_km`). The number of vehicles and the total number of km driven per year per vehicle should be integers, the power usage in kWh per km should be a float. The values for the power usage and the total number of km driven per year as given below for each vehicle type can be used as a benchmark. If you want to exclude a type of vehicle, you can set the number of vehicles to 0.

In [ ]:
example_vehicle_info = VehicleInfoDTO(
    vans=VehicleTypeInfoDTO(nr_of_vehicles=1, power_usage=0.2, annual_km=20000),
    boxtrucks=VehicleTypeInfoDTO(nr_of_vehicles=2, power_usage=0.5, annual_km=50000),
    semitrailertrucks=VehicleTypeInfoDTO(nr_of_vehicles=0, power_usage=1.0, annual_km=100000),
)

#### **Baseload preprocessing**

The baseload should be a list of `BaseloadProfile` objects to be given to the `EnergyDTO`. To obtain such a list, you can either use a csv-file following the specified format or select a business category and the annual consumption in kWh of the company, which can be converted to the specified format. In this example we select a business category and the annual power consumption of the company. The specified baseload profile format can be obtained using the following piece of code.

In [ ]:
example_baseload = get_baseload_profile_from_business_category(business_category="Farmer", annual_consumption=40000)

#### **Example EnergyDTO object**

Aside from the vehicle information and the baseload, you must configure additional variables in the `EnergyDTO` object. The arrival and departure time (`arrival_time` and `departure_time` respectively) should be strings following the format `'HH:MM'`, using the 24-hour clock. The zipcode should also be a string and can follow the Dutch zip code format (`'1111 AA'` or `'1111AA'`). The battery capacity in kWh should be an integer if you want to fix its capacity, but can be `None` if the battery capacity should result from the optimization. The charge point power can be provided as an integer. For a small consumer, you must provide the connection category as a string following the format `'{number_of_phases} x {ampere}A'`. For a large consumer, instead of the connection category, you must provide the contract capacity and the connection capacity, both in MVA.

In [ ]:
example_input_dto_small_consumer = EnergyDTOSmallConsumer(
    vehicle_info=example_vehicle_info,
    arrival_time="18:00",
    departure_time="08:00",
    baseload=example_baseload,
    zip_code="1111 AA",
    battery_capacity=None,
    charge_point_power=50,
    connection_category="3 x 80A",
)

In [ ]:
example_input_dto_large_consumer = EnergyDTOLargeConsumer(
    vehicle_info=example_vehicle_info,
    arrival_time="18:00",
    departure_time="08:00",
    baseload=example_baseload,
    zip_code="1111AA",
    battery_capacity=None,
    charge_point_power=50,
    contract_capacity=0.1,
    connection_capacity=0.2,
)

If you want to test the following results with the example input for the large consumer, you can easily switch out `example_input_dto_small_consumer` for `example_input_dto_large_consumer` in the code below. 

#### **Check and update current connection**

The first step is to see if the connection category is sufficient for the configuration as specified above. This check is not obligatory, but skipping this step could lead to the optimization failing and raising an error. Running this step returns an (updated) `EnergyDTO` object for the scenario's with and without EnergyPod separately and ensures that both configurations are sufficient.

In [ ]:
# Check if current connection fits and determine updated EnergyDTO objects for scenario's with and without EnergyPod
updated_energy_dtos = UpdateCurrentConnection(
    energy_dto=example_input_dto_small_consumer
).calculate_updated_energy_dtos()

# Updated EnergyDTO objects for scenario's with and without EnergyPod
without_ep_dto = updated_energy_dtos.without_ep_dto
with_ep_dto = updated_energy_dtos.with_ep_dto

#### **Calculate results for without EnergyPod scenario**

If you skipped the check and update current connection step, you can use the `EnergyDTO` as configured at the beginning of this file. Otherwise, you can use the updated `EnergyDTO` object for the without EnergyPod scenario from the update current connection step to compute the results. The results can be plotted and shown in a table. Example code on how to do so is shown below.

In [ ]:
# Get without EnergyPod results
without_energy_pod_results = calculate_results_without_energypod(without_energy_pod_input=without_ep_dto)

In [ ]:
# Calculate the grid tariff
if isinstance(without_ep_dto, EnergyDTOSmallConsumer):
    grid_tariff_without_ep = calculate_grid_tariff_small(
        grid_tariff_input=GridTariffDTOSmallConsumer(
            zip_code=without_ep_dto.zip_code, connection_category=without_ep_dto.connection_category
        )
    )
elif isinstance(without_ep_dto, EnergyDTOLargeConsumer):
    grid_tariff_without_ep = calculate_grid_tariff_large(
        grid_tariff_input=GridTariffDTOLargeConsumer(
            zip_code=without_ep_dto.zip_code,
            connection_capacity=without_ep_dto.connection_capacity,
            contract_capacity=without_ep_dto.contract_capacity,
            baseload=without_ep_dto.baseload,
        )
    )

In [ ]:
# Plot the results

# Total number of quarters of an hour in first week
nr_quarters = 96 * 7

# Extract baseload power from results
baseload_results = np.array([e.baseload_power for e in without_energy_pod_results.consumption])

# Extract contract power from results
contract_power_results = np.array([e.contract_power for e in without_energy_pod_results.consumption])

# Extract charge point power from results
cp_power_results = np.array([e.charge_point_power for e in without_energy_pod_results.consumption])

# Datetime labels for first week of the year
datetime_labels = []
for t in range(nr_quarters):
    if t % 24 == 0:
        datetime_labels.append(first_day_of_year + timedelta(minutes=t * 15))
    else:
        datetime_labels.append("")

# Plot the first week from results
x_ticks = np.arange(nr_quarters)
plt.figure(figsize=(25, 10))
plt.plot(contract_power_results[:nr_quarters], color="black", label="Grid connection")
plt.stackplot(
    np.array(x_ticks),
    np.array([baseload_results[:nr_quarters], cp_power_results[:nr_quarters]]),
    labels=["Baseload", "Charge Point Power"],
)
plt.xticks(ticks=x_ticks, labels=datetime_labels, rotation=90, size=14)
plt.yticks(size=14)
plt.ylim([0, np.max(baseload_results[:nr_quarters] + cp_power_results[:nr_quarters]) + 20])
plt.ylabel("Power (kW)", size=18)
plt.xlabel("Date", size=18)
plt.legend(fontsize="x-large", loc=2)
plt.show()

In [ ]:
# Show results in table format
results_df = pd.DataFrame(
    {
        "Result variable": [
            "Yearly total power exceeded",
            "Yearly costs for capacity exceedances",
            "Yearly charging demand not delivered",
            "Number of vehicles not fully charged per year",
            "Yearly costs for uncharged power",
            "Yearly energy costs",
            "Yearly fixed energy markup costs",
            "Yearly energy tax costs",
            "Yearly ERE savings",
            "Yearly grid tariff costs",
            "Total yearly costs",
            "Power of a single charge point",
            "Price of a single charge point per kW",
            "Total number of charge points",
            "Investment costs for the charge points",
        ],
        "Unit": ["kWh", "€", "kWh", "", "€", "€", "€", "€", "€", "€", "€", "kW", "€", "", "€"],
        "Value": [
            sum(without_energy_pod_results.power_capacity_exceedances_year),
            without_energy_pod_results.yearly_costs_capacity_exceedances,
            sum(without_energy_pod_results.uncharged_power),
            int(sum(without_energy_pod_results.vehicles_not_fully_charged)),
            without_energy_pod_results.yearly_costs_uncharged_power,
            without_energy_pod_results.total_yearly_energy_costs,
            without_energy_pod_results.energy_price_markup_costs,
            without_energy_pod_results.energy_tax_costs,
            without_energy_pod_results.ere_reduction_costs,
            grid_tariff_without_ep,
            without_energy_pod_results.total_yearly_energy_costs
            + without_energy_pod_results.energy_price_markup_costs
            + without_energy_pod_results.energy_tax_costs
            + grid_tariff_without_ep
            + without_energy_pod_results.yearly_costs_capacity_exceedances
            + without_energy_pod_results.yearly_costs_uncharged_power
            - without_energy_pod_results.ere_reduction_costs,
            example_input_dto_small_consumer.charge_point_power,
            cp_price_per_kw,
            example_vehicle_info.get_total_nr_of_vehicles(),
            without_energy_pod_results.investment_costs,
        ],
    }
)

results_df.set_index(["Result variable", "Unit", "Value"])

#### **Calculate results for with EnergyPod scenario**

If you skipped the check and update current connection step, you can use the `EnergyDTO` as configured at the beginning of this file. Otherwise, you can use the updated `EnergyDTO` object for the with EnergyPod scenario from the update current connection step to compute the results. The results can be plotted and shown in a table. Example code on how to do so is shown below.

In [ ]:
# With EnergyPod results
with_energy_pod_results = EnergyPodPower(energy_pod_input=with_ep_dto).calculate_energy_pod_power_results()

In [ ]:
# Calculate the grid tariff
if isinstance(with_ep_dto, EnergyDTOSmallConsumer):
    grid_tariff_with_ep = calculate_grid_tariff_small(
        grid_tariff_input=GridTariffDTOSmallConsumer(
            zip_code=with_ep_dto.zip_code, connection_category=with_ep_dto.connection_category
        )
    )
elif isinstance(with_ep_dto, EnergyDTOLargeConsumer):
    grid_tariff_with_ep = calculate_grid_tariff_large(
        grid_tariff_input=GridTariffDTOLargeConsumer(
            zip_code=with_ep_dto.zip_code,
            connection_capacity=with_ep_dto.connection_capacity,
            contract_capacity=with_ep_dto.contract_capacity,
            baseload=with_ep_dto.baseload,
        )
    )

In [ ]:
# Plot the optimization results

# Total number of quarters of an hour in first week
nr_quarters = 96 * 7

# Extract baseload power from results
baseload_results = np.array([e.baseload_power for e in with_energy_pod_results.consumption])

# Extract contract power from results
contract_power_results = np.array([e.contract_power for e in with_energy_pod_results.consumption])

# Extract charge point power from results
cp_power_results = np.array([e.charge_point_power for e in with_energy_pod_results.consumption])

# Extract battery power from results
bat_ch_power_results = np.array([e.battery_charge_power for e in with_energy_pod_results.consumption])
bat_dis_power_results = np.array([e.battery_discharge_power for e in with_energy_pod_results.consumption])

# Datetime labels for first week of the year
datetime_labels = []
for t in range(nr_quarters):
    if t % 24 == 0:
        datetime_labels.append(first_day_of_year + timedelta(minutes=t * 15))
    else:
        datetime_labels.append("")

# Plot the first week from results
x_ticks = np.arange(nr_quarters)
plt.figure(figsize=(25, 10))
plt.plot(contract_power_results[:nr_quarters], color="black", label="Grid connection")
plt.stackplot(
    np.array(x_ticks),
    np.array(
        [
            baseload_results[:nr_quarters],
            cp_power_results[:nr_quarters],
            bat_ch_power_results[:nr_quarters],
            bat_dis_power_results[:nr_quarters],
        ]
    ),
    labels=["Baseload", "Charge Point Power", "Battery Charge", "Battery Discharge"],
)
plt.xticks(ticks=x_ticks, labels=datetime_labels, rotation=90, size=14)
plt.yticks(size=14)
plt.ylim(
    [
        0,
        np.max(
            baseload_results[:nr_quarters]
            + cp_power_results[:nr_quarters]
            + bat_ch_power_results[:nr_quarters]
            + bat_dis_power_results[:nr_quarters]
        )
        + 20,
    ]
)
plt.ylabel("Power (kW)", size=18)
plt.xlabel("Date", size=18)
plt.legend(fontsize="x-large", loc=2)
plt.show()

In [ ]:
# Show results in table format
results_df = pd.DataFrame(
    {
        "Result variable": [
            "Yearly total power exceeded",
            "Yearly costs for capacity exceedances",
            "Yearly energy costs",
            "Yearly fixed energy markup costs",
            "Yearly energy tax costs",
            "Yearly ERE savings",
            "Yearly grid tariff costs",
            "Total yearly costs",
            "Energy volume of the battery",
            "Battery power",
            "Battery price per kWh",
            "Power of a single charge point",
            "Price of a single charge point per kW",
            "Total number of charge points",
            "Investment costs for the charge points",
            "Investment costs for the battery",
        ],
        "Unit": ["kWh", "€", "€", "€", "€", "€", "€", "€", "kWh", "kW", "€", "kW", "€", "", "€", "€"],
        "Value": [
            sum(with_energy_pod_results.power_capacity_exceedances_year),
            with_energy_pod_results.yearly_costs_capacity_exceedances,
            sum(with_energy_pod_results.yearly_energy_costs),
            with_energy_pod_results.energy_price_markup_costs,
            with_energy_pod_results.energy_tax_costs,
            with_energy_pod_results.ere_reduction_costs,
            grid_tariff_with_ep,
            sum(with_energy_pod_results.yearly_energy_costs)
            + with_energy_pod_results.energy_price_markup_costs
            + with_energy_pod_results.energy_tax_costs
            + grid_tariff_with_ep
            + with_energy_pod_results.yearly_costs_capacity_exceedances
            - with_energy_pod_results.ere_reduction_costs,
            with_energy_pod_results.battery_results.capacity,
            with_energy_pod_results.battery_results.power,
            battery_price_per_kwh,
            example_input_dto_small_consumer.charge_point_power,
            cp_price_per_kw,
            example_vehicle_info.get_total_nr_of_vehicles(),
            with_energy_pod_results.investment_costs_cp,
            with_energy_pod_results.battery_results.cost,
        ],
    }
)
results_df.set_index(["Result variable", "Unit", "Value"])

#### **Calculate yearly savings**

Using the total yearly costs for the scenario's with and without EnergyPod, you can compute the yearly savings. 

In [ ]:
# Yearly savings
total_yearly_costs_without_ep = (
    sum(without_energy_pod_results.yearly_costs)
    + without_energy_pod_results.energy_price_markup_costs
    + without_energy_pod_results.energy_tax_costs
    + without_energy_pod_results.yearly_costs_capacity_exceedances
    + without_energy_pod_results.yearly_costs_uncharged_power
    + grid_tariff_without_ep
    - without_energy_pod_results.ere_reduction_costs
)
total_yearly_costs_with_ep = (
    sum(with_energy_pod_results.yearly_energy_costs)
    + with_energy_pod_results.energy_price_markup_costs
    + with_energy_pod_results.energy_tax_costs
    + with_energy_pod_results.yearly_costs_capacity_exceedances
    + grid_tariff_with_ep
    - with_energy_pod_results.ere_reduction_costs
)
yearly_savings = calculate_yearly_savings(
    yearly_costs_no_ep=total_yearly_costs_without_ep, yearly_costs_ep=total_yearly_costs_with_ep
)

print(f"Yearly savings: €{yearly_savings}")  # noqa: T201

#### **Calculate payback time**

Using the yearly savings as calculated in the previous step and the investment costs for the scenario's with EnergyPod (investment costs for charge points and battery) and without EnergyPod (investment costs for charge points), you can compute the expected payback time.

In [ ]:
# Payback time
payback_time = calculate_payback_time(
    yearly_savings_ep=yearly_savings,
    investment_costs_no_ep=without_energy_pod_results.investment_costs,
    investment_costs_ep=with_energy_pod_results.investment_costs_cp + with_energy_pod_results.battery_results.cost,
)

print(f"Payback time: {payback_time} years")  # noqa: T201